# 3. CNN 성능 개선

이 노트북은 `02_CNN_구조_파악.ipynb`에서 만든 기본 CNN을 바탕으로, 어떤 기법을 추가하면 성능이 더 좋아지는지 MNIST 예제로 확인하는 실습입니다.

> GPU가 잡히지 않아 `Using device: cpu`가 나온다면 먼저 `00_Torch_GPU_셋업.ipynb`를 확인하세요.

## 목차

- [3-1. CNN 성능을 높이는 대표 방법](#3-1.-CNN-성능을-높이는-대표-방법)
- [3-2. 데이터 준비](#3-2.-데이터-준비)
- [3-3. 기본 CNN과 개선 CNN 비교](#3-3.-기본-CNN과-개선-CNN-비교)
- [3-4. 모델 학습 및 성능 비교](#3-4.-모델-학습-및-성능-비교)
- [3-5. 예측 결과 확인](#3-5.-예측-결과-확인)


## 3-1. CNN 성능을 높이는 대표 방법

기본 CNN도 MNIST에서는 충분히 잘 동작하지만, 실제 문제에서는 더 안정적이고 일반화가 잘 되는 구조가 필요합니다.

이번 노트북에서는 다음 기법을 사용합니다.

- **더 깊은 합성곱 구조**: 더 복잡한 특징을 학습할 수 있습니다.
- **Batch Normalization**: 학습을 안정화하고 수렴을 빠르게 만드는 데 도움이 됩니다.
- **Dropout**: 과적합을 줄이는 데 도움이 됩니다.
- **Data Augmentation**: 학습 데이터를 다양하게 만들어 일반화 성능을 높입니다.
- **Learning Rate Scheduler**: 학습이 진행될수록 학습률을 조절해 더 안정적으로 최적화합니다.

핵심은 단순히 층만 많이 쌓는 것이 아니라, **학습 안정성 + 일반화 성능 + 표현력**을 같이 개선하는 것입니다.


In [ ]:
# 필요한 라이브러리가 없다면 아래 주석을 해제해서 설치하세요.
# !pip install torch torchvision matplotlib


In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

torch.manual_seed(42)


## 3-2. 데이터 준비

이번에는 학습용 데이터에 약한 회전과 이동을 추가해 Data Augmentation을 적용합니다. 숫자 이미지가 조금 기울거나 위치가 달라져도 잘 분류하도록 돕기 위한 설정입니다.

또한 학습 데이터 일부를 validation 셋으로 분리해, 학습 중 모델 상태를 더 객관적으로 확인합니다.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_aug = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
full_train_eval = datasets.MNIST(root='./data', train=True, download=True, transform=eval_transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=eval_transform)

train_size = 55000
val_size = 5000
train_aug_subset, val_aug_subset = random_split(full_train_aug, [train_size, val_size])
train_eval_subset, val_eval_subset = random_split(full_train_eval, [train_size, val_size])

train_loader = DataLoader(train_aug_subset, batch_size=64, shuffle=True)
train_eval_loader = DataLoader(train_eval_subset, batch_size=1000, shuffle=False)
val_loader = DataLoader(val_eval_subset, batch_size=1000, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print('Train samples:', len(train_aug_subset))
print('Validation samples:', len(val_eval_subset))
print('Test samples:', len(test_dataset))


In [ ]:
images, labels = next(iter(train_loader))
print('배치 이미지 shape:', images.shape)
print('배치 라벨 shape:', labels.shape)

plt.figure(figsize=(10, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'label: {labels[i].item()}')
    plt.axis('off')
plt.tight_layout()
plt.show()


## 3-3. 기본 CNN과 개선 CNN 비교

먼저 `02`에서 사용한 기본 CNN을 다시 정의하고, 그 다음에 개선 CNN을 정의합니다.

개선 CNN의 차이점은 다음과 같습니다.

- 합성곱 블록 수 증가
- BatchNorm 추가
- Dropout 추가
- 분류기 구조 보강


In [ ]:
class BasicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.15),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.25)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


basic_model = BasicCNN()
improved_model = ImprovedCNN()

print('BasicCNN 파라미터 수:', f"{count_parameters(basic_model):,}")
print('ImprovedCNN 파라미터 수:', f"{count_parameters(improved_model):,}")


In [ ]:
print('BasicCNN 구조')
print(basic_model)
print()
print('ImprovedCNN 구조')
print(improved_model)


파라미터 수는 늘어났지만, 중요한 것은 단순한 증가가 아니라 **표현력과 일반화 성능을 같이 높이는 구조적 개선**이라는 점입니다.


## 3-4. 모델 학습 및 성능 비교

두 모델을 같은 데이터셋에서 학습시키고, validation/test 성능을 비교합니다.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def fit_model(model, train_loader, train_eval_loader, val_loader, criterion, optimizer, scheduler, device, epochs):
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(epochs):
        train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_loss, train_acc = evaluate(model, train_eval_loader, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if scheduler is not None:
            scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        current_lr = optimizer.param_groups[0]['lr']
        print(
            f'Epoch [{epoch + 1}/{epochs}] | '
            f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
            f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} | '
            f'LR: {current_lr:.6f}'
        )

    model.load_state_dict(best_state)
    return history, best_val_acc


In [ ]:
criterion = nn.CrossEntropyLoss()

basic_model = BasicCNN().to(device)
basic_optimizer = optim.Adam(basic_model.parameters(), lr=0.001)

improved_model = ImprovedCNN().to(device)
improved_optimizer = optim.Adam(improved_model.parameters(), lr=0.001)
improved_scheduler = optim.lr_scheduler.StepLR(improved_optimizer, step_size=3, gamma=0.5)


실습 시간을 고려해 epoch 수는 짧게 두었습니다. GPU를 사용할 수 있다면 `8 ~ 10 epoch` 정도로 늘려서 차이를 더 뚜렷하게 볼 수 있습니다.


In [ ]:
print('=== BasicCNN 학습 ===')
basic_history, basic_best_val_acc = fit_model(
    basic_model,
    train_loader,
    train_eval_loader,
    val_loader,
    criterion,
    basic_optimizer,
    scheduler=None,
    device=device,
    epochs=5
)

print()
print('=== ImprovedCNN 학습 ===')
improved_history, improved_best_val_acc = fit_model(
    improved_model,
    train_loader,
    train_eval_loader,
    val_loader,
    criterion,
    improved_optimizer,
    scheduler=improved_scheduler,
    device=device,
    epochs=5
)


In [ ]:
basic_test_loss, basic_test_acc = evaluate(basic_model, test_loader, criterion, device)
improved_test_loss, improved_test_acc = evaluate(improved_model, test_loader, criterion, device)

print('BasicCNN  Test Loss:', round(basic_test_loss, 4), '| Test Acc:', round(basic_test_acc, 4))
print('ImprovedCNN Test Loss:', round(improved_test_loss, 4), '| Test Acc:', round(improved_test_acc, 4))


In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(basic_history['val_loss'], label='Basic Val Loss')
plt.plot(improved_history['val_loss'], label='Improved Val Loss')
plt.title('Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(basic_history['val_acc'], label='Basic Val Acc')
plt.plot(improved_history['val_acc'], label='Improved Val Acc')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 3)
model_names = ['BasicCNN', 'ImprovedCNN']
test_accs = [basic_test_acc, improved_test_acc]
plt.bar(model_names, test_accs, color=['gray', 'steelblue'])
plt.ylim(0.9, 1.0)
plt.title('Test Accuracy Comparison')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()


일반적으로 다음과 같은 흐름이 보입니다.

- BasicCNN은 빠르게 수렴하지만, 더 복잡한 패턴을 배우는 데 한계가 있습니다.
- ImprovedCNN은 초반 학습이 조금 더 복잡하지만, validation/test 성능이 더 안정적으로 나오는 경우가 많습니다.
- BatchNorm과 Scheduler는 학습 안정성에, Dropout과 Augmentation은 일반화 성능에 기여합니다.


## 3-5. 예측 결과 확인

마지막으로 개선 CNN이 테스트 이미지에서 어떤 예측을 하는지 직접 확인합니다.


In [ ]:
improved_model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = improved_model(images)
    predictions = outputs.argmax(dim=1)

images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f'True: {labels[i].item()} / Pred: {predictions[i].item()}')
    plt.axis('off')

plt.tight_layout()
plt.show()


## 정리

이번 노트북에서 확인한 핵심은 다음과 같습니다.

- CNN 성능 개선은 단순히 층을 더 쌓는 것만으로 끝나지 않습니다.
- Data Augmentation은 모델이 다양한 입력 변화에 견고해지도록 돕습니다.
- BatchNorm은 학습을 더 안정적으로 만들고, Dropout은 과적합을 줄이는 데 도움을 줍니다.
- 더 깊고 정돈된 CNN 구조는 기본 CNN보다 더 높은 표현력을 가질 수 있습니다.
- Validation 셋과 Scheduler를 같이 사용하면 더 체계적으로 학습을 관리할 수 있습니다.

다음 단계에서는 실제 컬러 이미지 데이터셋(CIFAR-10 등)으로 넘어가 CNN이 더 큰 문제에서 어떻게 동작하는지 이어서 학습할 수 있습니다.
